# 05 - FedProx: meredam non-IID (TANPA SFM)

**Tujuan belajar:** memahami FedProx sebagai perbaikan FedAvg untuk data non-IID.

FedProx menambah **proximal term** pada loss lokal tiap klien:

    L_prox(W) = L_lokal(W) + (mu/2) * || W - W_global ||^2

Efeknya: update lokal tidak 'melenceng' terlalu jauh dari model global, sehingga
agregasi lebih stabil saat klien heterogen. Dengan `mu=0`, FedProx = FedAvg.

Kita pakai partisi non-IID (Dirichlet alpha kecil) dari nb04, lalu bandingkan
FedAvg (mu=0) vs FedProx (beberapa mu). Versi belajar dari
`02_survey-tanpaSFM/s06_fedprox_comm.ipynb`.

**Prasyarat:** `01_data_partition.ipynb`. Dijalankan di SageMaker.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scikit-learn
print('deps OK')

## 1. MLP + FedProx (proximal term) -- identik fed_node_survey.py


In [ ]:
import numpy as np
SEED=42
def init_weights(d, seed=42):
    r=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x):  return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; return _sig(a3).ravel(),(X,a1,z1,a2,z2,a3)
def train_local(W,X,y,epochs=2,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    # mu>0 -> proximal term FedProx: gradien += mu*(W - W_global)
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2,a3)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0);        gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(ws, sizes):
    tot=float(sum(sizes)); return [sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws))) for i in range(len(ws[0]))]
def predict(W,X): return forward(W,X)[0]
def zfit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zapp(X,mu,sd): return (X-mu)/sd
print('FedProx siap')

## 2. Muat populasi + partisi non-IID (Dirichlet alpha kecil)


In [ ]:
import os
DATA='./fl_data'
assert os.path.isdir(DATA), 'Jalankan 01_data_partition.ipynb dulu'
cs=sorted(set(int(f.split('_')[1]) for f in os.listdir(DATA) if f.startswith('client_')))
Xpop=np.vstack([np.load(f'{DATA}/client_{i}_train.npz')['X'] for i in cs])
ypop=np.concatenate([np.load(f'{DATA}/client_{i}_train.npz')['y'] for i in cs])
g=np.load(f'{DATA}/global_test.npz'); Xg,yg=g['X'],g['y']; d=Xg.shape[1]; K=4
def dirichlet_partition(y,K,alpha,seed=42):
    r=np.random.RandomState(seed); out=[[] for _ in range(K)]
    for c in np.unique(y):
        ci=np.where(y==c)[0]; r.shuffle(ci)
        cuts=(np.cumsum(r.dirichlet([alpha]*K))*len(ci)).astype(int)[:-1]
        for k,part in enumerate(np.split(ci,cuts)): out[k]+=part.tolist()
    return [np.array(sorted(ix)) for ix in out]
ALPHA=0.1   # non-IID cukup berat agar efek FedProx terlihat
parts=dirichlet_partition(ypop,K,ALPHA,SEED)
print('attack%% per klien (alpha=%.1f):'%ALPHA, [round(100*ypop[ix].mean(),1) for ix in parts])

## 3. Bandingkan FedAvg (mu=0) vs FedProx (mu>0)

Partisi klien SAMA untuk semua mu -> perbandingan adil, hanya proximal term yang beda.


In [ ]:
from sklearn.metrics import matthews_corrcoef
cli=[]
for ix in parts:
    Xi,yi=Xpop[ix],ypop[ix]; m_,s_=zfit(Xi)
    cli.append(dict(Xz=zapp(Xi,m_,s_), y=yi, mu=m_, sd=s_, n=len(yi)))
def run(mu, rounds=30, local_epochs=2):
    W=init_weights(d,SEED)
    for t in range(rounds):
        cw=[]; sz=[]
        for c in cli:
            if c['n']<10: continue
            cw.append(train_local(W,c['Xz'],c['y'],epochs=local_epochs,mu=mu,W_global=W,seed=SEED)); sz.append(c['n'])
        W=fedavg(cw,sz)
    ms=[matthews_corrcoef(yg,(predict(W,zapp(Xg,c['mu'],c['sd']))>=0.5).astype(int)) for c in cli]
    return float(np.mean(ms))

MUS=[0.0, 0.01, 0.1, 1.0]   # 0.0 = FedAvg
res=[]
for mu in MUS:
    m=run(mu); res.append((mu,m))
    tag='FedAvg' if mu==0 else f'FedProx(mu={mu})'
    print(f'{tag:18s}: MCC global = {m:.4f}')

## 4. Simpan + kesimpulan belajar


In [ ]:
import csv
os.makedirs('learn_out',exist_ok=True)
with open('learn_out/05_fedprox.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mu','mcc_global']); [w.writerow([mu,round(m,4)]) for mu,m in res]
fedavg_mcc=res[0][1]; best_mu,best_mcc=max(res,key=lambda r:r[1])
print('tersimpan: learn_out/05_fedprox.csv')
print(f'FedAvg (mu=0) MCC={fedavg_mcc:.4f}  |  terbaik: mu={best_mu} MCC={best_mcc:.4f} (delta {best_mcc-fedavg_mcc:+.4f})')
print('\nSelesai jalur belajar FL tanpa-SFM: partisi -> baseline -> FedAvg -> non-IID -> FedProx.')
print('Berikutnya: lihat 02_survey-tanpaSFM/ untuk versi riset (cross-dataset, multi-seed CI, adversarial).')